# Amazon Bedrock AgentCore PostgreSQL agent demonstration

We start by importing the required libraries: `boto3` to invoke the agent on Bedrock AgentCore,
`json` to serialize the request payload and deserialize the response, `uuid` to generate unique session identifiers, and `IPython.display` to render the agent's responses as formatted Markdown in the notebook.

In [1]:
# Import the required libraries
import json
import boto3
import uuid
from IPython.display import Markdown, display

Next, we set up the client and helper functions. We define the AgentCore
Runtime ARN and region, create a `boto3` client for invoking the
agent, and implement three helper functions:
`parse_streaming_response` to extract agent messages from the response
stream, `get_streaming_response` to invoke the agent and return the
parsed messages, and `print_messages` to render a conversation turn in
the notebook with formatted text, tool calls and tool results.

In [2]:
# Configure the Bedrock AgentCore runtime
AGENTCORE_RUNTIME_ARN = "<agentcore-runtime-arn>"
REGION = "<agentcore-runtime-region>"

# Create the Bedrock AgentCore client
agentcore_client = boto3.client(
    service_name="bedrock-agentcore",
    region_name=REGION
)

def parse_streaming_response(response: dict) -> list[dict]:
    """
    Parse the streaming response from AgentCore and extract agent messages.

    Parameters:
    ===============================================================================
    response: dict
        The raw response dictionary returned by invoke_agent_runtime,
        containing a StreamingBody object under the "response" key.

    Returns:
    ===============================================================================
    list[dict]
        List of agent message dictionaries extracted from the event stream.
    """
    messages = []
    for line in response["response"].iter_lines():
        if line:
            data = line.decode("utf-8")
            try:
                start, end = data.index("{"), data.rindex("}")
                event = json.loads(data[start:end + 1], strict=False)
                if "message" in event:
                    messages.append(event["message"])
            except (ValueError, json.JSONDecodeError):
                pass
    return messages


def get_streaming_response(prompt: str, session_id: str) -> list[dict]:
    """
    Invoke the forecasting agent on Amazon Bedrock AgentCore and return
    the agent messages.

    Parameters:
    ===============================================================================
    prompt: str
        The user message to send to the agent.

    session_id: str
        The session identifier used to maintain conversation context across turns.

    Returns:
    ===============================================================================
    list[dict]
        List of agent message dictionaries.
    """
    # Invoke the agent on Bedrock AgentCore
    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENTCORE_RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}),
    )
    
    # Parse and return the streaming response
    messages = parse_streaming_response(response)
    return messages


def print_messages(prompt: str, messages: list[dict]) -> None:
    """
    Display a conversation turn in a Jupyter notebook, including the user
    prompt, agent text responses, tool calls and tool results.

    Parameters:
    ===============================================================================
    prompt: str
        The user message sent to the agent.

    messages: list[dict]
        List of agent message dictionaries returned by the agent.
    """
    # Display the user prompt
    display(Markdown(f"<h2>User:</h2><br>{prompt}"))

    # Display the agent response
    display(Markdown("<h2>Agent:</h2><br>"))
    for message in messages:
        for content in message["content"]:
            if "text" in content:
                # Display text response as formatted markdown
                display(Markdown(content["text"]))
            elif "toolUse" in content:
                # Display tool call name and inputs
                display(Markdown(f"🔨 Ran `{content['toolUse']['name']}`"))
                print({"input": content["toolUse"]["input"]})
            elif "toolResult" in content:
                # Display tool result output
                if "content" in content["toolResult"]:
                    tool_output = content["toolResult"]["content"][0]["text"]
                    display(Markdown("🔨 Output:"))
                    print({"output": tool_output})

We generate a unique session ID to identify this conversation. The same
session ID will be reused across all turns to maintain conversation
context and short-term memory.

In [3]:
# Generate the session ID
session_id = str(uuid.uuid4())

We start the conversation by asking the agent what it can do.

In [4]:
# Turn 1 / 4
prompt = """
What can you help me with?
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
What can you help me with?


<h2>Agent:</h2><br>

I'm your PostgreSQL database assistant! Here's what I can help you with:

---

## 🔍 Schema Exploration
- **Browse your database**: List schemas, tables, views, sequences, and extensions
- **Inspect objects**: Get detailed info on table columns, data types, constraints, indexes, and more

---

## 📝 SQL Query Generation & Execution
- **Answer questions** about your data by writing and running SQL queries
- **Translate plain English** into accurate SQL — just ask something like *"How many orders were placed last month?"*
- **Run custom SQL** queries you provide directly

---

## ⚡ Performance & Optimization
- **Explain query plans**: Understand how PostgreSQL executes a query and identify bottlenecks
- **Index recommendations**: Analyze specific queries or your overall workload and suggest optimal indexes
- **Top queries**: Find the slowest or most resource-intensive queries in your system

---

## 🏥 Database Health
- **Full health checks** covering:
  - 🔗 Connections & utilization
  - 🗂️ Index health (invalid, duplicate, bloated indexes)
  - 🧹 Vacuum & transaction ID wraparound
  - 🔄 Replication lag & slots
  - 💾 Buffer cache hit rates
  - 🔢 Sequence overflow risks
  - ✅ Constraint validity

---

## 💡 Example Questions You Can Ask
- *"Show me the tables in the public schema"*
- *"How many users signed up this week?"*
- *"What are the slowest queries in my database?"*
- *"Is my database healthy?"*
- *"Recommend indexes for this query: SELECT ..."*

---

**Just ask your question in plain English, and I'll take it from there!** 🚀

We then ask the agent to list the tables in the database. 

In [5]:
# Turn 2 / 4
prompt = """
What tables are in the database?
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
What tables are in the database?


<h2>Agent:</h2><br>

Let me look up all the schemas and their tables for you!

🔨 Ran `list_schemas`

{'input': {}}


...

🔨 Ran `list_objects`

{'input': {'schema_name': 'public', 'object_type': 'table'}}


🔨 Output:

{'output': "[{'schema': 'public', 'name': 'products', 'type': 'BASE TABLE'}, {'schema': 'public', 'name': 'sales', 'type': 'BASE TABLE'}]"}


Your database has **1 user schema (`public`)** with **2 tables**:

| # | Table Name | Type |
|---|------------|------|
| 1 | **products** | BASE TABLE |
| 2 | **sales** | BASE TABLE |

Would you like me to:
- 🔎 **Inspect the details** of either table (columns, data types, indexes, constraints)?
- 📊 **Query the data** — e.g., row counts, sample rows, or something more specific?

Next, we ask the agent a question which requires memory of the previous turn.

In [6]:
# Turn 3 / 4
prompt = """
Show me the data in the second table.
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
Show me the data in the second table.


<h2>Agent:</h2><br>

Let me first check the structure of the **sales** table, then query the data — all at once!

🔨 Ran `get_object_details`

{'input': {'schema_name': 'public', 'object_name': 'sales'}}


...

🔨 Output:

{'output': "{'basic': {'schema': 'public', 'name': 'sales', 'type': 'table'}, 'columns': [{'column': 'product_id', 'data_type': 'character varying', 'is_nullable': 'YES', 'default': None}, {'column': 'units', 'data_type': 'integer', 'is_nullable': 'YES', 'default': None}], 'constraints': [{'name': 'sales_product_id_fkey', 'type': 'FOREIGN KEY', 'columns': ['product_id']}], 'indexes': []}"}


🔨 Output:

{'output': "[{'product_id': '1', 'units': 10}, {'product_id': '2', 'units': 20}, {'product_id': '3', 'units': 30}]"}


Here's the data in the **sales** table:

| product_id | units |
|------------|-------|
| 1 | 10 |
| 2 | 20 |
| 3 | 30 |

**Table Details:**
- **Columns:** `product_id` (varchar), `units` (integer)
- **Constraints:** `product_id` is a **foreign key** referencing the `products` table
- **Rows returned:** 3

Would you like me to **join this with the products table** to see more details, or do any further analysis on this data?

Finally, we ask the agent to join the two tables and compute a simple derived metric.

In [7]:
# Turn 4 / 4
prompt = """
Combine the two tables and add the percentage of total sales for each product.
"""
messages = get_streaming_response(prompt, session_id)
print_messages(prompt, messages)

<h2>User:</h2><br>
Combine the two tables and add the percentage of total sales for each product.


<h2>Agent:</h2><br>

🔨 Ran `get_object_details`

{'input': {'schema_name': 'public', 'object_name': 'products'}}


🔨 Output:

...

🔨 Ran `execute_sql`

{'input': {'sql': '\nSELECT \n    p.product_id,\n    p.product_name,\n    s.units,\n    ROUND(s.units * 100.0 / SUM(s.units) OVER (), 2) AS percentage_of_total_sales\nFROM public.products p\nJOIN public.sales s ON p.product_id = s.product_id\nORDER BY p.product_id;\n'}}


🔨 Output:

{'output': "[{'product_id': '1', 'product_name': 'Product A', 'units': 10, 'percentage_of_total_sales': Decimal('16.67')}, {'product_id': '2', 'product_name': 'Product B', 'units': 20, 'percentage_of_total_sales': Decimal('33.33')}, {'product_id': '3', 'product_name': 'Product C', 'units': 30, 'percentage_of_total_sales': Decimal('50.00')}]"}


Here's the combined table with the percentage of total sales for each product:

| Product ID | Product Name | Units | % of Total Sales |
|------------|--------------|-------|-----------------|
| 1 | Product A | 10 | 16.67% |
| 2 | Product B | 20 | 33.33% |
| 3 | Product C | 30 | 50.00% |
| | **Total** | **60** | **100%** |

**Key Insights:**
- 🥇 **Product C** is the top seller, accounting for **half of all sales (50%)**
- 🥈 **Product B** comes in second at **33.33%**
- 🥉 **Product A** has the smallest share at **16.67%**

Would you like any further analysis, such as visualizations, filtering, or additional metrics?